In [ ]:
!pip install -U transformers

In [ ]:
!pip install python-dotenv

## Local Inference on GPU
Model page: https://huggingface.co/deepseek-ai/deepseek-coder-1.3b-base

⚠️ If the generated code snippets do not work, please open an issue on either the [model repo](https://huggingface.co/deepseek-ai/deepseek-coder-1.3b-base)
			and/or on [huggingface.js](https://github.com/huggingface/huggingface.js/blob/main/packages/tasks/src/model-libraries-snippets.ts) 🙏

In [ ]:
# Use a pipeline as a high-level helper
from transformers import pipeline

pipe = pipeline("text-generation", model="deepseek-ai/deepseek-coder-1.3b-base")

In [ ]:
# Load model directly
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("deepseek-ai/deepseek-coder-1.3b-base")
model = AutoModelForCausalLM.from_pretrained("deepseek-ai/deepseek-coder-1.3b-base")

In [2]:
pip install llama-cpp-python  --no-cache-dir --upgrade


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.6/50.6 MB 189.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 268.8 MB/s eta 0:00:00
  Created wheel for llama-cpp-python: filename=llama_cpp_python-0.3.15-cp311-cp311-linux_x86_64.whl size=4480278 sha256=76a8cbba390ae645ee29592017a4a10fc10812b7e70887c3f53636026b1a147d
  Stored in directory: /tmp/pip-ephem-wheel-cache-yi3bk1bo/wheels/34/62/22/63039c8c4c8da7ff68e81b8357e64c199faff9df9a2b5e5e3b
Successfully built llama-cpp-python


In [ ]:
from llama_cpp import Llama

# Load the model only once when module is imported
llm = Llama(
    model_path="stablelm-zephyr-3b.Q4_0.gguf",  # Change path as needed
    n_ctx=4096,
    n_threads=4,
    n_gpu_layers=0
)

def ask_question(question: str) -> str:
    prompt = f"""### Instruction:
    I'm learning DSA from scratch. Give me explaination in detail for the following question.

### Question:
{question}

### Answer:"""

    response = llm(
        prompt,
        max_tokens=1024,
        temperature=0.7,
        top_p=0.95,
        stop=["###"]
    )

    return response["choices"][0]["text"].strip()
def solve_question(question: str) -> str:
    prompt = f"""### Instruction:
    I'm learning DSA from scratch. Solve the following question while explaining each step.

### Question:
{question}

### Answer:"""

    response = llm(
        prompt,
        max_tokens=1024,
        temperature=0.7,
        top_p=0.95,
        stop=["###"]
    )

    return response["choices"][0]["text"].strip()


In [5]:
import json
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# Load JSON files
with open("syllabus_topic_data.json", "r") as f:
    syllabus_dict = json.load(f)

with open("question_for_pattern.json", "r") as f:
    pattern_questions = json.load(f)

with open("question_for_solving.json", "r") as f:
    solving_questions = json.load(f)

# Load DeepSeek model on CPU to avoid OOM
tokenizer = AutoTokenizer.from_pretrained("deepseek-ai/deepseek-coder-1.3b-base")
model = AutoModelForCausalLM.from_pretrained("deepseek-ai/deepseek-coder-1.3b-base").to("cpu")

# Syllabus table
syllabus_table = """
=== DAA Syllabus ===

Module | Hours | Topics
--- | --- | ---
Module 1: Algorithm Analysis | 8 | Time complexity, recurrence relations, asymptotic notations
Module 2: Linear Data Structures | 7 | Arrays, Stack, Queue, Linked List, Polynomial manipulation
Module 3: Searching and Sorting | 7 | Merge Sort, Quick Sort, Binary Search, Counting Sort
Module 4: Trees | 8 | Binary Trees, BST, Traversals, 2-3 Trees
Module 5: Heaps and AVL Trees | 5 | Heaps, AVL Rotations, Priority Queue
Module 6: Graphs | 6 | BFS, DFS, Prim’s, Kruskal’s, Dijkstra
Module 7: Hashing | 4 | Hash functions, Separate Chaining, Open Addressing
"""

# Normalize for case/space/plural
def normalize(text):
    return text.lower().rstrip('s').replace("-", "").replace("_", "").strip()

# Recursively find topic
def find_nested_topic(data, user_input):
    user_input = normalize(user_input)
    if isinstance(data, dict):
        for k, v in data.items():
            if normalize(k) == user_input:
                return v
            result = find_nested_topic(v, user_input)
            if result:
                return result
    return None

# Recursively navigate nested dictionary to find exact path
def navigate_nested_dict(data):
    path = []
    while isinstance(data, dict):
        print("\nAvailable subtopics:")
        subtopics = list(data.keys())
        for sub in subtopics:
            print(f"🔹 {sub}")
        topic = input("Enter the topic you want to explore (or type 'exit', 'quit', or 'stop'): ").strip()
        if normalize(topic) in {"exit", "quit", "stop"}:
            return None, None
        matches = [k for k in data if normalize(k) == normalize(topic)]
        if matches:
            key = matches[0]
            path.append(key)
            data = data[key]
        else:
            print("❌ Topic not found. Please try again.")
            return None, None
    return data, path[-1] if path else None

# Ask for explanation using DeepSeek model
def explain_with_deepseek(topic_name):
    prompt = f"Explain the DSA topic: {topic_name}"
    inputs = tokenizer(prompt, return_tensors="pt").to("cpu")
    outputs = model.generate(**inputs, max_new_tokens=300)
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    print("\n📘 Explanation:")
    print(response.split(":", 1)[-1].strip())

# Main driver
print("Device set to use:", "cuda:0" if torch.cuda.is_available() else "cpu")
past_queries = set()

while True:
    user_input = input("Ask your question about dsa or type 'exit', 'quit', or 'stop': ").strip().lower()
    if user_input in {"exit", "quit", "stop"}:
        print("Session ended. Goodbye!")
        break
    if user_input in past_queries:
        print("You've already asked that. Please try a different question.")
        continue
    past_queries.add(user_input)

    print(syllabus_table)

    while True:
        print("\nWhat would you like next?")
        print("1️⃣ Detailed portions of a particular topic")
        print("2️⃣ Sample questions to understand the pattern")
        print("3️⃣ Sample questions to practice")

        next_input = input("Choose option (1/2/3), or type 'exit', 'quit', or 'stop': ").strip().lower()
        if next_input in {"exit", "quit", "stop"}:
            print("Session ended. Goodbye!")
            exit()

        elif next_input == "1":
            topic = input("Enter the topic you want details for (or type 'exit', 'quit', or 'stop'): ").strip()
            if normalize(topic) in {"exit", "quit", "stop"}:
                print("Session ended. Goodbye!")
                exit()
            result = find_nested_topic(syllabus_dict, topic)
            if result:
                print(f"\n=== Detailed Breakdown: {topic.title()} ===")
                print(json.dumps(result, indent=2))
                ask_expl = input("📘 Do you want explanation for any topic from the above? (yes/no): ").strip().lower()
                if ask_expl == "yes":
                    chosen = input("Please enter the topic you want explained from the above listed topics: ").strip()
                    explain_with_deepseek(chosen)
            else:
                print("❌ Sorry, that topic was not found.")

        elif next_input == "2":
            topic = input("Enter the topic for sample questions to understand the pattern (or type 'exit', 'quit', or 'stop'): ").strip()
            if normalize(topic) in {"exit", "quit", "stop"}:
                print("Session ended. Goodbye!")
                exit()
            data = find_nested_topic(pattern_questions, topic)
            if data:
                final_topic_data, actual_topic = navigate_nested_dict(data)
                if final_topic_data:
                    print("\n=== Sample Pattern Questions ===")
                    for q in final_topic_data:
                        print(f"🔹 {q}")
                    ask = input(f"\n📘 Would you like an explanation for the topic '{actual_topic}'? (yes/no): ").strip().lower()
                    if ask == "yes":
                        explain_with_deepseek(actual_topic)
            else:
                print("❌ Sorry, that topic was not found.")

        elif next_input == "3":
            topic = input("Enter the topic for practice questions (or type 'exit', 'quit', or 'stop'): ").strip()
            if normalize(topic) in {"exit", "quit", "stop"}:
                print("Session ended. Goodbye!")
                exit()
            data = find_nested_topic(solving_questions, topic)
            if data:
                final_topic_data, actual_topic = navigate_nested_dict(data)
                if final_topic_data:
                    print("\n=== Sample Practice Questions ===")
                    for q in final_topic_data:
                        print(f"📝 {q}")
                    ask = input(f"\n📘 Would you like an explanation for the topic '{actual_topic}'? (yes/no): ").strip().lower()
                    if ask == "yes":
                        explain_with_deepseek(actual_topic)
            else:
                print("❌ Sorry, that topic was not found.")

        else:
            print("❓ Invalid option. Please choose 1, 2, 3 or an exit command.")


FileNotFoundError: [Errno 2] No such file or directory: 'syllabus_topic_data.json'

In [ ]:
!apt-get install -y cmake build-essential
!CMAKE_ARGS="-DLLAMA_CUBLAS=on" FORCE_CMAKE=1 pip install llama-cpp-python --no-cache-dir


Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
build-essential is already the newest version (12.9ubuntu3).
cmake is already the newest version (3.22.1-1ubuntu1.22.04.2).
0 upgraded, 0 newly installed, 0 to remove and 35 not upgraded.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.6/50.6 MB 284.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 206.9 MB/s eta 0:00:00
  error: subprocess-exited-with-error
  
  × Building wheel for llama-cpp-python (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for llama-cpp-python
Failed to build llama-cpp-python
ERROR: ERROR: Failed to build in

In [6]:
import json
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from llama_cpp import Llama

import subprocess

# def run_llama(prompt):
#     result = subprocess.run(
#         ["./main", "-m", "stablelm-zephyr-3b.Q4_0.gguf", "-p", prompt],
#         stdout=subprocess.PIPE,
#         stderr=subprocess.DEVNULL,  # hides llama.cpp logs
#         text=True
#     )
#     return result.stdout.strip()

# 5️⃣ Example usage
# output = run_llama("Hello, how are you?")
# print(output)

# === Load Local LLM ===
llm = Llama(
    model_path="/content/stablelm-zephyr-3b.Q4_0.gguf",  # Update this path as needed
    n_ctx=4096,
    n_threads=4,
    n_gpu_layers=0
)

def ask_question(question: str) -> str:
    prompt = f"""### Instruction:
I'm learning DSA from scratch. Give me explanation in detail for the following question.

### Question:
{question}

### Answer:"""
    response = llm(prompt, max_tokens=1024, temperature=0.7, top_p=0.95, stop=["###"])
    return response["choices"][0]["text"].strip()

def solve_question(question: str) -> str:
    prompt = f"""### Instruction:
I'm learning DSA from scratch. Solve the following question while explaining each step.

### Question:
{question}

### Answer:"""
    response = llm(prompt, max_tokens=1024, temperature=0.7, top_p=0.95, stop=["###"])
    return response["choices"][0]["text"].strip()

# === Load JSON files ===
with open("syllabus_topic_data.json", "r") as f:
    syllabus_dict = json.load(f)

with open("question_for_pattern.json", "r") as f:
    pattern_questions = json.load(f)

with open("question_for_solving.json", "r") as f:
    solving_questions = json.load(f)

# === Static syllabus table ===
syllabus_table = """
=== DAA Syllabus ===

Module | Hours | Topics
--- | --- | ---
Module 1: Algorithm Analysis | 8 | Time complexity, recurrence relations, asymptotic notations
Module 2: Linear Data Structures | 7 | Arrays, Stack, Queue, Linked List, Polynomial manipulation
Module 3: Searching and Sorting | 7 | Merge Sort, Quick Sort, Binary Search, Counting Sort
Module 4: Trees | 8 | Binary Trees, BST, Traversals, 2-3 Trees
Module 5: Heaps and AVL Trees | 5 | Heaps, AVL Rotations, Priority Queue
Module 6: Graphs | 6 | BFS, DFS, Prim’s, Kruskal’s, Dijkstra
Module 7: Hashing | 4 | Hash functions, Separate Chaining, Open Addressing
"""

# === Helper functions ===
def find_nested_topic(data, user_input):
    user_input = user_input.lower().rstrip('s')
    if isinstance(data, dict):
        for k, v in data.items():
            if user_input == k.lower().rstrip('s'):
                return v
            result = find_nested_topic(v, user_input)
            if result:
                return result
    return None

def navigate_nested_dict(data):
    while isinstance(data, dict):
        print("\nAvailable subtopics:")
        for sub in data.keys():
            print(f"🔹 {sub}")
        topic = input("Enter the topic you want to explore (or type 'exit', 'quit', or 'stop'): ").strip().lower()
        if topic in {"exit", "quit", "stop"}:
            return None, None
        matches = [k for k in data.keys() if topic == k.lower() or topic == k.lower().rstrip('s')]
        if matches:
            selected_key = matches[0]
            data = data[selected_key]
        else:
            print("❌ Topic not found. Please try again.")
            return None, None
    return data, selected_key

# === Main loop ===
print("Device set to use:", "cuda:0" if torch.cuda.is_available() else "cpu")
past_queries = set()

while True:
    user_input = input("Ask your question about dsa or type 'exit', 'quit', or 'stop': ").strip().lower()
    if user_input in {"exit", "quit", "stop"}:
        print("Session ended. Goodbye!")
        break

    if user_input in past_queries:
        print("You've already asked that. Please try a different question.")
        continue
    past_queries.add(user_input)

    print(syllabus_table)

    while True:
        print("\nWhat would you like next?")
        print("1️⃣ Detailed portions of a particular topic")
        print("2️⃣ Sample questions to understand the pattern")
        print("3️⃣ Sample questions to practice")

        next_input = input("Choose option (1/2/3), or type 'exit', 'quit', or 'stop': ").strip().lower()

        if next_input in {"exit", "quit", "stop"}:
            print("Session ended. Goodbye!")
            break

        elif next_input == "1":
            topic = input("Enter the topic you want details for (or type 'exit', 'quit', or 'stop'): ").strip().lower()
            if topic in {"exit", "quit", "stop"}:
                print("Session ended. Goodbye!")
                break
            result = find_nested_topic(syllabus_dict, topic)
            if result:
                print(f"\n=== Detailed Breakdown: {topic.title()} ===")
                print(json.dumps(result, indent=2))

                follow_up = input("\n📘 Do you want explanation for any topic from the above? (yes/no): ").strip().lower()
                if follow_up == "yes":
                    explain_topic = input("Please enter the topic you want explained from the above listed topics: ").strip()
                    print("\n📘 Explanation:")
                    print(ask_question(explain_topic))
            else:
                print("❌ Sorry, that topic was not found.")

        elif next_input == "2":
            topic = input("Enter the topic for sample questions to understand the pattern (or type 'exit', 'quit', or 'stop'): ").strip().lower()
            if topic in {"exit", "quit", "stop"}:
                print("Session ended. Goodbye!")
                break
            data = find_nested_topic(pattern_questions, topic)
            if data:
                final_topic_data, selected_topic = navigate_nested_dict(data)
                if final_topic_data:
                    print("\n=== Sample Pattern Questions ===")
                    for q in final_topic_data:
                        print(f"🔹 {q}")
                    follow_up = input(f"\n📘 Do you want an explanation for the topic '{selected_topic}'? (yes/no): ").strip().lower()
                    if follow_up == "yes":
                        print("\n📘 Explanation:")
                        print(ask_question(selected_topic))
            else:
                print("❌ Sorry, that topic was not found.")

        elif next_input == "3":
            topic = input("Enter the topic you want practice questions for (or type 'exit', 'quit', or 'stop'): ").strip().lower()
            if topic in {"exit", "quit", "stop"}:
                print("Session ended. Goodbye!")
                break
            data = find_nested_topic(solving_questions, topic)
            if data:
                final_topic_data, selected_topic = navigate_nested_dict(data)
                if final_topic_data:
                    print("\n=== Practice Questions ===")
                    for q in final_topic_data:
                        print(f"🔸 {q}")
                    follow_up = input(f"\n📘 Do you want help solving the topic '{selected_topic}'? (yes/no): ").strip().lower()
                    if follow_up == "yes":
                        print("\n🧠 Solving Explanation:")
                        print(solve_question(selected_topic))
            else:
                print("❌ Sorry, that topic was not found.")

        else:
            print("❓ Invalid option. Please choose 1, 2, 3 or an exit command.")


ValueError: Model path does not exist: /content/stablelm-zephyr-3b.Q4_0.gguf